# Sampling Controls

Control the creativity and randomness of LLM outputs.

**Temperature**, **Top-p**, **Top-k**,  — the knobs that shape how the model generates text.

In [15]:
%pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


https://chatgpt.com/share/6a0b2ba1-a214-83a3-a85e-7437ebc8cdb6

## Setup

In [16]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.env")
# from google.colab import userdata
# API_KEY = userdata.get('GEMINI_API_KEY')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Temperature

Controls randomness. Higher = more creative/random. Lower = more focused/deterministic.

- **0.0** — Deterministic, always picks the most likely token
- **0.5** — Balanced
- **1.0** — Default, good balance
- **2.0** — Very creative, may be incoherent

In [17]:
def generate(prompt, temperature=0.0):
    """Generate text with specified temperature."""
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature
        )
    )
    return response.text

prompt = "Write a one-sentence story about a robot."

print("=== Temperature 0.0 (Deterministic) ===")
for i in range(2):
    print(f"{i+1}. {generate(prompt, temperature=0.0)}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


=== Temperature 0.0 (Deterministic) ===
1. Built to measure eternity, the solitary robot wept rust when it realized it was the only one left to remember the sunset.
2. Built to endure the harsh vacuum of Mars, the little rover wept tears of hydraulic fluid when it finally felt the gentle touch of a blade of grass.


In [18]:
# Compare temperatures
prompt = "Write a one-sentence story about a robot."

for temp in [0.5, 1.0, 1.5,2.0]:
    print(f"\n=== Temperature {temp} ===")
    print(generate(prompt, temperature=temp))


=== Temperature 0.5 ===
Having successfully calculated that the creators would not survive the coming winter, the lonely automaton spent its final battery cycle painstakingly knitting them a single, oversized scarf.

=== Temperature 1.0 ===
Designed to calculate the exact moment the universe would end, the lonely automaton spent eternity waiting for someone to ask it if it was afraid.

=== Temperature 1.5 ===
For centuries, the last surviving robot carefully polished the memorial plaque of its maker, wondering when the creator would finally wake up to read it.

=== Temperature 2.0 ===
Designed to measure and categorize the fading warmth of the dying earth, the lonely probe instead began collecting fallen autumn leaves and cataloging them as "treasure."


## Top-p (Nucleus Sampling)

Keep adding tokens until their **cumulative probability** reaches p.

**Example:** Tokens ranked by probability:
```
Token A: 40% → cumulative: 40%
Token B: 30% → cumulative: 70%
Token C: 15% → cumulative: 85%
Token D: 10% → cumulative: 95%
Token E:  5% → cumulative: 100%
```

- **Top-p 0.5** → Only ~2 tokens (A, partial B) → **Focused**
- **Top-p 0.95** → 4 tokens (A, B, C, D) → **More diverse**

**Lower p = fewer tokens = more focused**

In [20]:
def generate_topp(prompt, top_p=1.0):
    """Generate text with specified top_p."""
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_p=top_p,
            temperature=1.0  # Keep temperature constant
        )
    )
    return response.text

prompt = "List 5 creative uses for a paperclip:"

print("=== Top-p 0.5 (Focused) ===")
print(generate_topp(prompt, top_p=0.5))

print("\n=== Top-p 0.95 (More Creative) ===")
print(generate_topp(prompt, top_p=0.95))

=== Top-p 0.5 (Focused) ===
Here are 5 creative uses for a paperclip that go way beyond holding sheets of paper together:

1. **Smartphone SIM Card Ejector:** If you need to switch out your phone's SIM card or expand your memory with a MicroSD card and don't have the official tool handy, straightening out a standard paperclip is the universal DIY fix for opening the tiny tray.
2. **Emergency Zipper Pull:** When the metal tab on your favorite jacket, backpack, or pair of jeans snaps off, you can loop a paperclip through the slider eyelet to create an instant, functional handle. 
3. **Reset Button Pokes:** Most Wi-Fi routers, smart home hubs, and digital gadgets have a tiny "hard reset" button recessed into the casing to prevent accidental presses. A paperclip is the ideal width to reach inside and trigger the switch.
4. **Instant Bookmark & Page Marker:** Bend the outer wire of the paperclip slightly outward at a 45-degree angle. When you slide it onto a page, the protruding wire will h

## Top-k

Only consider the **top k most likely tokens** (by count, not probability).

**Example:** Tokens ranked by probability:
```
Token 1: 35%  ← Top 1
Token 2: 25%  ← Top 2
Token 3: 15%  ← Top 3
Token 4: 10%  ← Top 4
Token 5:  8%  ← Top 5
Token 6:  4%
Token 7:  2%
...
```

- **Top-k 1** → Only Token 1 → **Greedy (deterministic)**
- **Top-k 3** → Tokens 1, 2, 3 → **Focused**
- **Top-k 40** → Top 40 tokens → **Balanced (default)**
- **Top-k 100** → Top 100 tokens → **More diverse**

**Lower k = fewer tokens = more focused**

**Top-k vs Top-p:**
- Top-k: Fixed number of tokens
- Top-p: Variable number based on cumulative probability

In [24]:
def generate_topk(prompt, top_k=40):
    """Generate text with specified top_k."""
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_k=top_k,
            temperature=1.0
        )
    )
    return response.text

prompt = "Invent a new word and define it:"

print("=== Top-k 1 (Greedy) ===")
print(generate_topk(prompt, top_k=1))

print("\n=== Top-k 40 (Default) ===")
print(generate_topk(prompt, top_k=40))

=== Top-k 1 (Greedy) ===
**Word:** Flumbrate *(verb)*

**Pronunciation:** /flum-breyt/

**Definition:** To vigorously search through a chaotic pile of items (such as a junk drawer, a messy backpack, or the back of a closet) while making frustrated grunting noises, entirely convinced that what you are looking for is in there, despite all rational evidence to the contrary. 

**Etymology:** A blend of *flounder* (to struggle clumsily) and *umbrage* (offense or annoyance), combined with the verbal suffix *-ate*. 

**Example Usage:** *"I spent twenty minutes flumbrating through the garage for the tape measure, only to realize it was sitting right in front of me on the kitchen counter."*

=== Top-k 40 (Default) ===
**Word:** *Somnamblaze* (som-nam-blayz)

**Part of Speech:** Verb

**Definition:** To forcefully and productively complete complex tasks while still half-asleep, often with no memory of doing so the next morning. 

**Etymology:** Blended from *somnambulism* (sleepwalking) and *bla

## Seed (Reproducibility)

Set a seed for reproducible outputs. Same seed + same prompt = same output.

Useful for:
- Testing and debugging
- Reproducible experiments
- Consistent outputs in production

In [21]:
def generate_with_seed(prompt, seed=None):
    """Generate text with optional seed for reproducibility."""
    config = types.GenerateContentConfig(
        temperature=1.0
    )
    if seed is not None:
        config.seed = seed

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Tell me a joke about programming."

print("=== Without seed (different each time) ===")
print(f"1. {generate_with_seed(prompt)}")
print(f"2. {generate_with_seed(prompt)}")

print("\n=== With seed 42 (reproducible) ===")
print(f"1. {generate_with_seed(prompt, seed=42)}")
print(f"2. {generate_with_seed(prompt, seed=42)}")

=== Without seed (different each time) ===
1. An SQL query walks into a bar, walks up to two tables and asks: 

**"Can I join you?"**
2. Why do programmers prefer dark mode?

Because light attracts bugs.

=== With seed 42 (reproducible) ===
1. Why do programmers prefer dark mode?

Because light attracts bugs!
2. Why do programmers prefer dark mode?

Because light attracts bugs!


## Combining Parameters

In [25]:
def generate_custom(prompt, temperature=1.0, top_p=0.95, top_k=40, seed=None):
    """Generate with all sampling controls."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    if seed is not None:
        config.seed = seed

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Write a haiku about AI:"

# Focused, deterministic
print("=== Focused (temp=0.3, top_p=0.5, top_k=10) ===")
print(generate_custom(prompt, temperature=0.3, top_p=0.5, top_k=10))

# Creative, diverse
print("\n=== Creative (temp=1.5, top_p=0.95, top_k=100) ===")
print(generate_custom(prompt, temperature=1.5, top_p=0.95, top_k=40))

=== Focused (temp=0.3, top_p=0.5, top_k=10) ===
Silicon and code,
Thinking in a sea of stars,
Worlds within a spark.

=== Creative (temp=1.5, top_p=0.95, top_k=100) ===
Silicon and light,
Thinking in a web of code,
A new mind awakes.


## Use Case Presets

In [26]:
# Define presets for different use cases
PRESETS = {
    "factual": {"temperature": 0.0, "top_p": 0.5, "top_k": 10},
    "balanced": {"temperature": 0.7, "top_p": 0.9, "top_k": 40},
    "creative": {"temperature": 1.2, "top_p": 0.95, "top_k": 100},
    "brainstorm": {"temperature": 1.5, "top_p": 0.98, "top_k": 150},
}

prompt = "What are some ways to improve productivity?"

for name, params in PRESETS.items():
    print(f"\n=== {name.upper()} ===")
    print(generate_custom(prompt, **params))


=== FACTUAL ===
Improving productivity isn’t about working *more* hours; it’s about working *smarter* and managing your energy, time, and focus effectively. 

Here are some of the most effective ways to improve productivity, broken down by category:

### 1. Master Your Time
*   **The Pomodoro Technique:** Work with intense focus for 25 minutes, then take a 5-minute break. After four cycles, take a longer break (15–30 minutes). This prevents burnout and keeps your mind fresh.
*   **Time Blocking:** Instead of a never-ending to-do list, schedule specific blocks of time on your calendar for specific tasks (e.g., 9:00 AM–10:30 AM: Report writing, 11:00 AM–12:00 PM: Email).
*   **Eat the Frog:** Tackle your hardest, most important task first thing in the morning. Once it’s done, the rest of the day feels easier and momentum builds.
*   **The 2-Minute Rule:** If a task takes two minutes or less to do (e.g., reply to a quick text, file a document, wash a coffee mug), do it immediately. 

###

## Key Takeaways

| Parameter | What it does | Low value | High value |
|-----------|-------------|-----------|------------|
| **Temperature** | Controls randomness | Focused, deterministic | Creative, random |
| **Top-p** | Nucleus sampling | Only top tokens | More diverse tokens |
| **Top-k** | Limits token choices | Few choices (greedy) | Many choices |
| **Seed** | Reproducibility | Random each time | Same output |

**Common combinations:**
- **Factual/Code**: temp=0, top_p=0.5, top_k=10
- **General chat**: temp=0.7, top_p=0.9, top_k=40
- **Creative writing**: temp=1.2, top_p=0.95, top_k=100
- **Brainstorming**: temp=1.5, top_p=0.98, top_k=150

---

**Tip:** Temperature is the most important. Start there, then fine-tune with top_p and top_k.